# Mechanism of Signal-Chaos Decoupling

## Research Question

The refined synthesis (notebook 10) established that the transmission peak $|C(t^*)|$ is
invariant across sparsity $p$, even as quantum chaos (measured by $\langle r \rangle$) is
destroyed. But **what controls the signal if not chaos?**

### Hypotheses

- **H0 (Coupling dominates)**: The L-R coupling strength $\mu$ alone determines the
  transmission peak. The three $H(\mu)$ curves at $p = 1.0, 0.1, 0.05$ should overlay
  within statistical error for all $\mu$.

- **H1 (Chaos modulates)**: Internal chaos modulates the signal. At some $\mu$ values
  (likely small $\mu$ where the signal is weak), the $H(\mu)$ curves should separate
  between chaotic and non-chaotic systems.

### Physical Motivation

The traversable wormhole protocol couples left and right copies via
$V = i\mu \sum_j \psi^L_j \psi^R_j$. The transmission signal
$C(t) = \langle \text{TFD} | \psi^R_j(t) \psi^L_j(0) | \text{TFD} \rangle$
probes how information propagates through this coupling. If the signal depends
only on $\mu$ (H0), it means the coupling itself creates the information channel
regardless of whether the internal dynamics are chaotic. This would undermine
claims that the signal evidences holographic/gravitational physics.

## Experimental Design

| Parameter | Value | Rationale |
|-----------|-------|----------|
| $N$ per side | 10 | Largest feasible for full eigendecomposition |
| $\beta$ | 8.0 | Strong TFD entanglement |
| Sparsity $p$ | 1.0, 0.1, 0.05 | Chaotic ($\langle r\rangle \approx 0.59$), edge ($\approx 0.57$), non-chaotic ($\approx 0.39$) |
| $\mu$ values | 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50 | Full dynamic range from weak to strong coupling |
| Realizations | 50 per $(p, \mu)$ point | Non-negotiable for SEM $\lesssim 0.01$ |
| Time range | $[0, 50]$, 200 points | Captures peak even at small $\mu$ |
| Total instances | $3 \times 8 \times 50 = 1200$ | |

### Optimization

For each $(p, \text{seed})$ pair, the `DoubledSYK` system and TFD state are built once,
then the 8 $\mu$ values are swept by calling `build_coupled_hamiltonian(mu)` repeatedly.
This reduces the computation from 1200 to 150 unique system constructions.

### Analyses

| Label | Analysis | Purpose |
|-------|----------|---------|
| A1 | $H(\mu)$ vs $\mu$ at three $p$ | Primary test: do curves overlay? |
| A2 | One-way ANOVA at each $\mu$ | Statistical test: $p$-value for $p$-dependence |
| A3 | Functional fit $H(\mu) = \tanh(\alpha \mu^\gamma)$ | Does a single function describe all three? |
| A4 | Peak time $t^*(\mu, p)$ | Side observable: does timing also decouple? |
| A5 | Peak width (FWHM) vs $\mu$ | Side observable: does width depend on chaos? |
| A6 | Level spacing verification | Confirm chaos regimes before interpreting |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import h5py
import os
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.optimize import curve_fit

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 13, 'axes.titlesize': 13,
    'legend.fontsize': 9, 'figure.dpi': 150,
})

base = '..'
data_path = os.path.join(base, 'data', 'research', 'mu_mechanism.h5')
results_dir = os.path.join(base, 'results')
os.makedirs(results_dir, exist_ok=True)

print('Mu mechanism analysis notebook loaded.')

## 3. Data Loading

In [ ]:
with h5py.File(data_path, 'r') as f:
    # Metadata
    N_per_side = int(f.attrs['N_per_side'])
    beta = float(f.attrs['beta'])
    sparsity_values = list(f.attrs['sparsity_values'])
    mu_values = list(f.attrs['mu_values'])
    n_realizations = int(f.attrs['n_realizations'])
    total_time = float(f.attrs['total_compute_time_s'])
    failures = int(f.attrs['failures'])
    timestamp = str(f.attrs['timestamp'])
    
    # Data arrays
    peak_heights = np.array(f['peak_heights'])  # (n_sparsity, n_mu, n_realizations)
    peak_times = np.array(f['peak_times'])
    peak_fwhms = np.array(f['peak_fwhms'])
    r_values = np.array(f['r_values'])  # (n_sparsity, n_realizations)
    t_array = np.array(f['t_array'])

n_sparsity = len(sparsity_values)
n_mu = len(mu_values)

print(f'Data loaded from: {data_path}')
print(f'  Timestamp: {timestamp}')
print(f'  N_per_side = {N_per_side} (doubled dim = {2**N_per_side})')
print(f'  beta = {beta}')
print(f'  Sparsity values: {sparsity_values}')
print(f'  Mu values: {mu_values}')
print(f'  Realizations: {n_realizations}')
print(f'  Total instances: {n_sparsity * n_mu * n_realizations}')
print(f'  Failures: {failures}')
print(f'  Compute time: {total_time/3600:.1f} hours')
print()
print(f'  peak_heights: shape {peak_heights.shape}, NaN count: {np.sum(np.isnan(peak_heights))}')
print(f'  peak_times: shape {peak_times.shape}, NaN count: {np.sum(np.isnan(peak_times))}')
print(f'  peak_fwhms: shape {peak_fwhms.shape}, NaN count: {np.sum(np.isnan(peak_fwhms))}')
print(f'  r_values: shape {r_values.shape}, NaN count: {np.sum(np.isnan(r_values))}')

## 4. A6: Verification of Chaos Regimes

Before interpreting the mu sweep, confirm that the three sparsity levels produce
distinct chaos regimes. This uses the level spacing ratio $\langle r \rangle$
computed from single-copy SYK eigenvalues embedded in the sweep data.

| Regime | $p$ | Expected $\langle r \rangle$ | Reference |
|--------|-----|-----------------------------|-----------|
| Chaotic (GUE) | 1.0 | $\approx 0.60$ | Wigner-Dyson |
| Edge of chaos | 0.1 | $\approx 0.57$ | Near GUE |
| Non-chaotic (Poisson) | 0.05 | $\approx 0.39$ | Level repulsion lost |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

labels_p = {1.0: 'Chaotic (GUE)', 0.1: 'Edge of chaos', 0.05: 'Non-chaotic'}
colors_p = {1.0: 'C0', 0.1: 'C1', 0.05: 'C3'}

# Panel A: Histograms of <r> by sparsity
ax = axes[0]
for ip, p in enumerate(sparsity_values):
    rv = r_values[ip, :]
    ax.hist(rv, bins=15, alpha=0.5, color=colors_p[p], density=True,
            label=f'p={p} ({labels_p[p]})')
    ax.axvline(np.mean(rv), color=colors_p[p], linestyle='-', linewidth=2)
ax.axvline(0.6027, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='GUE (0.603)')
ax.axvline(0.3863, color='gray', linestyle='--', linewidth=1.5, alpha=0.6, label='Poisson (0.386)')
ax.set_xlabel(r'$\langle r \rangle$ per realization')
ax.set_ylabel('Density')
ax.set_title('(a) Level Spacing Distributions')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.2)

# Panel B: Mean <r> with error bars
ax = axes[1]
r_means = [np.mean(r_values[ip, :]) for ip in range(n_sparsity)]
r_sems = [np.std(r_values[ip, :]) / np.sqrt(n_realizations) for ip in range(n_sparsity)]
r_stds = [np.std(r_values[ip, :]) for ip in range(n_sparsity)]

bar_colors = [colors_p[p] for p in sparsity_values]
x_pos = np.arange(n_sparsity)
ax.bar(x_pos, r_means, yerr=r_sems, capsize=8, color=bar_colors, alpha=0.8,
       edgecolor='black', linewidth=0.5)
ax.axhline(0.6027, color='red', linestyle='--', alpha=0.5, label='GUE')
ax.axhline(0.3863, color='gray', linestyle='--', alpha=0.5, label='Poisson')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'p={p}' for p in sparsity_values])
ax.set_ylabel(r'$\langle r \rangle$')
ax.set_title('(b) Mean Level Spacing Ratio')
ax.set_ylim(0, 0.75)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

# Add text annotations
for i, (m, s) in enumerate(zip(r_means, r_sems)):
    ax.text(i, m + s + 0.02, f'{m:.4f}\n({labels_p[sparsity_values[i]].split(" ")[0]})',
            ha='center', fontsize=8)

# Panel C: Separation test
ax = axes[2]
# Compute pairwise t-tests
pairs = [(0, 1), (0, 2), (1, 2)]
pair_labels = []
t_stats = []
p_vals_ttest = []
for i, j in pairs:
    t_stat, p_val = stats.ttest_ind(r_values[i, :], r_values[j, :])
    pair_labels.append(f'p={sparsity_values[i]} vs p={sparsity_values[j]}')
    t_stats.append(abs(t_stat))
    p_vals_ttest.append(p_val)

x_pair = np.arange(len(pairs))
bars = ax.bar(x_pair, [-np.log10(pv) for pv in p_vals_ttest], color=['C0', 'C2', 'C4'], alpha=0.8)
ax.axhline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.axhline(-np.log10(0.001), color='darkred', linestyle='--', alpha=0.5, label='p=0.001')
ax.set_xticks(x_pair)
ax.set_xticklabels(pair_labels, fontsize=7, rotation=15)
ax.set_ylabel(r'$-\log_{10}(p_{\mathrm{value}})$')
ax.set_title('(c) Pairwise Separation (t-test)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('A6: Chaos Regime Verification (50 realizations per sparsity)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, '11_A6_chaos_verification.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print('\n=== A6: Chaos Regime Verification ===')
print(f'{"p":>6} {"<r>":>8} {"SEM":>8} {"std":>8} {"Regime":<20}')
print('-' * 55)
for ip, p in enumerate(sparsity_values):
    rv = r_values[ip, :]
    print(f'{p:>6.2f} {np.mean(rv):>8.4f} {np.std(rv)/np.sqrt(len(rv)):>8.4f} '
          f'{np.std(rv):>8.4f} {labels_p[p]:<20}')

print('\nPairwise t-tests:')
for label, t, pv in zip(pair_labels, t_stats, p_vals_ttest):
    sig = '***' if pv < 0.001 else ('**' if pv < 0.01 else ('*' if pv < 0.05 else 'ns'))
    print(f'  {label}: t={t:.2f}, p={pv:.2e} {sig}')

# PASS/FAIL
all_separated = all(pv < 0.05 for pv in p_vals_ttest)
chaotic_ok = r_means[0] > 0.55
nonchaotic_ok = r_means[2] < 0.45
print(f'\nVerification: {"PASS" if (all_separated and chaotic_ok and nonchaotic_ok) else "FAIL"}')
print(f'  All pairs separated (p<0.05): {all_separated}')
print(f'  p=1.0 chaotic (<r> > 0.55): {chaotic_ok} (<r>={r_means[0]:.4f})')
print(f'  p=0.05 non-chaotic (<r> < 0.45): {nonchaotic_ok} (<r>={r_means[2]:.4f})')

### A6 Verdict

The three sparsity levels produce distinct chaos regimes:
- $p = 1.0$: $\langle r \rangle \approx 0.589$ (GUE, chaotic)
- $p = 0.1$: $\langle r \rangle \approx 0.568$ (near GUE, edge of chaos)
- $p = 0.05$: $\langle r \rangle \approx 0.394$ (Poisson, non-chaotic)

The key separations are clear: both $p = 1.0$ and $p = 0.1$ vs $p = 0.05$
are highly significant ($p < 10^{-7}$). The $p = 1.0$ vs $p = 0.1$ comparison
is marginal ($p \approx 0.14$), reflecting that both are still chaotic systems.
The experiment successfully probes two genuinely chaotic and one genuinely
non-chaotic regime. Proceed to the mu sweep analysis.

## 5. Core Analysis

### A1: $H(\mu)$ vs $\mu$ at Three Sparsities

**This is the primary figure.** If H0 holds, the three curves overlay.
If H1 holds, they separate (especially at small $\mu$).

In [ ]:
# Compute means and SEMs
h_means = np.zeros((n_sparsity, n_mu))
h_sems = np.zeros((n_sparsity, n_mu))
h_stds = np.zeros((n_sparsity, n_mu))

for ip in range(n_sparsity):
    for imu in range(n_mu):
        vals = peak_heights[ip, imu, :]
        h_means[ip, imu] = np.mean(vals)
        h_stds[ip, imu] = np.std(vals)
        h_sems[ip, imu] = h_stds[ip, imu] / np.sqrt(n_realizations)

# A1: Main figure
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

markers = {1.0: 'o', 0.1: 's', 0.05: '^'}

# Panel A: H(mu) vs mu with error bars
ax = axes[0]
for ip, p in enumerate(sparsity_values):
    ax.errorbar(mu_values, h_means[ip, :], yerr=h_sems[ip, :],
                fmt=f'{markers[p]}-', capsize=4, linewidth=2.5, markersize=8,
                color=colors_p[p], label=f'p={p} ({labels_p[p]})')

ax.set_xlabel(r'Coupling strength $\mu$')
ax.set_ylabel(r'Peak height $H(\mu)$')
ax.set_title(r'(a) $H(\mu)$ vs $\mu$: Three Sparsities')
ax.legend(fontsize=8, loc='lower right')
ax.set_ylim(0.3, 1.05)
ax.grid(True, alpha=0.3)

# Panel B: Zoomed in at low mu where separation might appear
ax = axes[1]
low_mu_idx = [i for i, m in enumerate(mu_values) if m <= 0.10]
for ip, p in enumerate(sparsity_values):
    ax.errorbar([mu_values[i] for i in low_mu_idx],
                [h_means[ip, i] for i in low_mu_idx],
                yerr=[h_sems[ip, i] for i in low_mu_idx],
                fmt=f'{markers[p]}-', capsize=5, linewidth=2.5, markersize=9,
                color=colors_p[p], label=f'p={p}')
    # Show +/- 1 std band
    low_means = np.array([h_means[ip, i] for i in low_mu_idx])
    low_stds = np.array([h_stds[ip, i] for i in low_mu_idx])
    ax.fill_between([mu_values[i] for i in low_mu_idx],
                    low_means - low_stds, low_means + low_stds,
                    alpha=0.1, color=colors_p[p])

ax.set_xlabel(r'Coupling strength $\mu$')
ax.set_ylabel(r'Peak height $H(\mu)$')
ax.set_title(r'(b) Zoomed: $\mu \leq 0.10$ (1$\sigma$ bands)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel C: Ratio to dense (p=1.0)
ax = axes[2]
for ip, p in enumerate(sparsity_values):
    if ip == 0:
        continue  # skip p=1.0 (ratio is identically 1)
    ratios = h_means[ip, :] / h_means[0, :]
    # Error propagation for ratio
    ratio_err = ratios * np.sqrt((h_sems[ip, :] / h_means[ip, :])**2 +
                                  (h_sems[0, :] / h_means[0, :])**2)
    ax.errorbar(mu_values, ratios, yerr=ratio_err,
                fmt=f'{markers[p]}-', capsize=4, linewidth=2.5, markersize=8,
                color=colors_p[p], label=f'p={p} / p=1.0')

ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.axhline(0.99, color='gray', linestyle=':', alpha=0.3)
ax.axhline(1.01, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel(r'Coupling strength $\mu$')
ax.set_ylabel(r'$H(\mu, p) \;/\; H(\mu, p{=}1)$')
ax.set_title('(c) Ratio to Dense')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.95, 1.08)

plt.suptitle(r'A1: Transmission Peak $H(\mu)$ at Three Chaos Regimes (50 realizations)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, '11_A1_H_mu_primary.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print the data table
print('\n=== A1: H(mu, p) Data Table ===')
print(f'{"mu":>6}', end='')
for p in sparsity_values:
    print(f'  p={p:.2f} (mean +/- SEM)', end='')
print()
print('-' * 75)
for imu, mu in enumerate(mu_values):
    print(f'{mu:>6.2f}', end='')
    for ip in range(n_sparsity):
        print(f'  {h_means[ip, imu]:.4f} +/- {h_sems[ip, imu]:.4f}   ', end='')
    print()

print('\nMax absolute difference between any two sparsities:')
for imu, mu in enumerate(mu_values):
    max_diff = max(abs(h_means[i, imu] - h_means[j, imu])
                   for i in range(n_sparsity) for j in range(i+1, n_sparsity))
    pooled_sem = np.sqrt(sum(h_sems[ip, imu]**2 for ip in range(n_sparsity)) / n_sparsity)
    z = max_diff / pooled_sem if pooled_sem > 0 else 0
    print(f'  mu={mu:.2f}: max|diff| = {max_diff:.4f}, z = {z:.1f}')

### A2: ANOVA at Each $\mu$

For each $\mu$, test whether the three sparsity groups have significantly different
peak heights using one-way ANOVA. Under H0 (coupling dominates), all $F$-tests
should be non-significant.

In [ ]:
# ANOVA at each mu
anova_results = []
for imu, mu in enumerate(mu_values):
    groups = [peak_heights[ip, imu, :] for ip in range(n_sparsity)]
    F_stat, p_val = stats.f_oneway(*groups)
    anova_results.append({'mu': mu, 'F': F_stat, 'p': p_val})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: F-statistic vs mu
ax = axes[0]
mus = [r['mu'] for r in anova_results]
Fs = [r['F'] for r in anova_results]
ps = [r['p'] for r in anova_results]

ax.bar(range(len(mus)), Fs, color='C0', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(mus)))
ax.set_xticklabels([f'{m:.2f}' for m in mus])
ax.set_xlabel(r'$\mu$')
ax.set_ylabel('F-statistic')
ax.set_title('(a) ANOVA F-statistic at Each $\mu$')
# Critical F for df1=2, df2=147 at alpha=0.05
F_crit = stats.f.ppf(0.95, 2, 3 * n_realizations - 3)
ax.axhline(F_crit, color='red', linestyle='--', alpha=0.5,
           label=f'F_crit(0.05) = {F_crit:.2f}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

# Panel B: p-values vs mu
ax = axes[1]
ax.bar(range(len(mus)), [-np.log10(p) for p in ps], color='C1', alpha=0.8,
       edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(mus)))
ax.set_xticklabels([f'{m:.2f}' for m in mus])
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$-\log_{10}(p)$')
ax.set_title('(b) ANOVA p-value at Each $\mu$')
ax.axhline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p = 0.05')
ax.axhline(-np.log10(0.05 / len(mus)), color='darkred', linestyle='--', alpha=0.5,
           label=f'Bonferroni (p = {0.05/len(mus):.4f})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('A2: One-Way ANOVA Testing Sparsity Dependence at Each $\mu$',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, '11_A2_anova.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print table
print('\n=== A2: ANOVA Results ===')
print(f'{"mu":>6} {"F":>8} {"p-value":>12} {"Significant?":>14}')
print('-' * 45)
bonferroni = 0.05 / len(mus)
any_significant = False
for r in anova_results:
    sig = 'YES' if r['p'] < 0.05 else 'no'
    bonf = ' (Bonferroni)' if r['p'] < bonferroni else ''
    if r['p'] < 0.05:
        any_significant = True
    print(f'{r["mu"]:>6.2f} {r["F"]:>8.3f} {r["p"]:>12.4e} {sig:>14}{bonf}')

print(f'\nBonferroni threshold: p < {bonferroni:.4f}')
if not any_significant:
    print('Result: No mu value shows significant sparsity dependence -> H0 supported')
else:
    n_sig = sum(1 for r in anova_results if r['p'] < 0.05)
    n_bonf = sum(1 for r in anova_results if r['p'] < bonferroni)
    print(f'Result: {n_sig}/{len(mus)} significant at p<0.05, {n_bonf}/{len(mus)} survive Bonferroni')

### A3: Functional Fit [RETRACTED]

**RETRACTION NOTICE**: The tanh fit $H(\mu) = \tanh(\alpha \mu^\gamma)$ was found during
audit to have chi2/dof ranging from 38 to 1785, meaning the model is statistically
rejected at all sparsity levels. The fit parameters are unreliable and should not be
interpreted. The code below is preserved for transparency but the results are not valid.

See also: `12_fwhm_corrected.ipynb` for the corrected analysis.

In [ ]:
def model_tanh(mu, alpha, gamma):
    """H(mu) = tanh(alpha * mu^gamma)"""
    return np.tanh(alpha * np.power(mu, gamma))

def model_power_sat(mu, a, b, c):
    """H(mu) = 1 - a * exp(-b * mu^c)"""
    return 1.0 - a * np.exp(-b * np.power(mu, c))

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

mu_fine = np.linspace(0.01, 0.55, 200)
fit_params_tanh = {}
fit_params_powersat = {}

# Panel A: Tanh fits
ax = axes[0]
for ip, p in enumerate(sparsity_values):
    ax.errorbar(mu_values, h_means[ip, :], yerr=h_sems[ip, :],
                fmt=markers[p], capsize=4, markersize=8, color=colors_p[p],
                label=f'p={p} (data)', zorder=5)
    try:
        popt, pcov = curve_fit(model_tanh, mu_values, h_means[ip, :],
                               p0=[5.0, 0.5], sigma=h_sems[ip, :],
                               absolute_sigma=True, maxfev=10000)
        perr = np.sqrt(np.diag(pcov))
        fit_params_tanh[p] = {'alpha': popt[0], 'gamma': popt[1],
                              'alpha_err': perr[0], 'gamma_err': perr[1]}
        y_fit = model_tanh(mu_fine, *popt)
        ax.plot(mu_fine, y_fit, '--', color=colors_p[p], linewidth=1.5, alpha=0.7)
        # Residuals
        y_pred = model_tanh(np.array(mu_values), *popt)
        chi2 = np.sum(((h_means[ip, :] - y_pred) / h_sems[ip, :])**2)
        fit_params_tanh[p]['chi2'] = chi2
        fit_params_tanh[p]['chi2_dof'] = chi2 / (n_mu - 2)
    except RuntimeError:
        fit_params_tanh[p] = None

ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$H(\mu)$')
ax.set_title(r'(a) Fit: $H = \tanh(\alpha \mu^\gamma)$')
ax.legend(fontsize=7)
ax.set_ylim(0.3, 1.05)
ax.grid(True, alpha=0.3)

# Panel B: Fit parameter comparison
ax = axes[1]
alphas = [fit_params_tanh[p]['alpha'] for p in sparsity_values if fit_params_tanh[p]]
alpha_errs = [fit_params_tanh[p]['alpha_err'] for p in sparsity_values if fit_params_tanh[p]]
gammas = [fit_params_tanh[p]['gamma'] for p in sparsity_values if fit_params_tanh[p]]
gamma_errs = [fit_params_tanh[p]['gamma_err'] for p in sparsity_values if fit_params_tanh[p]]

x_fit = np.arange(n_sparsity)
ax.errorbar(x_fit - 0.1, alphas, yerr=alpha_errs, fmt='o', capsize=6,
            markersize=10, color='C0', label=r'$\alpha$')
ax2 = ax.twinx()
ax2.errorbar(x_fit + 0.1, gammas, yerr=gamma_errs, fmt='s', capsize=6,
             markersize=10, color='C1', label=r'$\gamma$')
ax.set_xticks(x_fit)
ax.set_xticklabels([f'p={p}' for p in sparsity_values])
ax.set_ylabel(r'$\alpha$', color='C0')
ax2.set_ylabel(r'$\gamma$', color='C1')
ax.set_title('(b) Fit Parameters by Sparsity')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
ax.grid(True, alpha=0.2)

# Panel C: Residuals
ax = axes[2]
for ip, p in enumerate(sparsity_values):
    if fit_params_tanh[p] is None:
        continue
    alpha_fit = fit_params_tanh[p]['alpha']
    gamma_fit = fit_params_tanh[p]['gamma']
    y_pred = model_tanh(np.array(mu_values), alpha_fit, gamma_fit)
    residuals = (h_means[ip, :] - y_pred) / h_sems[ip, :]
    ax.plot(mu_values, residuals, f'{markers[p]}-', color=colors_p[p],
            linewidth=1.5, markersize=7, label=f'p={p}')

ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(2, color='red', linestyle=':', alpha=0.3)
ax.axhline(-2, color='red', linestyle=':', alpha=0.3)
ax.set_xlabel(r'$\mu$')
ax.set_ylabel('Normalized residual')
ax.set_title('(c) Fit Residuals (in SEM units)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(r'A3: Functional Fit $H(\mu) = \tanh(\alpha \mu^\gamma)$',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, '11_A3_functional_fit.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print fit parameters
print('\n=== A3: Fit Parameters ===')
print(f'{"p":>6} {"alpha":>10} {"gamma":>10} {"chi2/dof":>10}')
print('-' * 40)
for p in sparsity_values:
    fp = fit_params_tanh[p]
    if fp:
        print(f'{p:>6.2f} {fp["alpha"]:>8.3f}+/-{fp["alpha_err"]:.3f}'
              f' {fp["gamma"]:>8.4f}+/-{fp["gamma_err"]:.4f}'
              f' {fp["chi2_dof"]:>10.2f}')

# Check parameter consistency
if all(fit_params_tanh[p] is not None for p in sparsity_values):
    alpha_range = max(alphas) - min(alphas)
    alpha_avg_err = np.mean(alpha_errs)
    gamma_range = max(gammas) - min(gammas)
    gamma_avg_err = np.mean(gamma_errs)
    print(f'\nalpha range: {alpha_range:.3f} (avg uncertainty: {alpha_avg_err:.3f})')
    print(f'gamma range: {gamma_range:.4f} (avg uncertainty: {gamma_avg_err:.4f})')
    alpha_consistent = alpha_range < 3 * alpha_avg_err
    gamma_consistent = gamma_range < 3 * gamma_avg_err
    print(f'alpha consistent within 3-sigma: {alpha_consistent}')
    print(f'gamma consistent within 3-sigma: {gamma_consistent}')

## 6. Side Observable Analyses

### A4: Peak Time $t^*(\mu, p)$

The peak time $t^*$ is the time at which $|C(t)|$ reaches its maximum.
If the coupling $\mu$ controls the timescale, $t^*$ should depend on $\mu$
but not on $p$.

In [ ]:
# Compute peak time statistics
t_means = np.zeros((n_sparsity, n_mu))
t_sems = np.zeros((n_sparsity, n_mu))
t_stds = np.zeros((n_sparsity, n_mu))

for ip in range(n_sparsity):
    for imu in range(n_mu):
        vals = peak_times[ip, imu, :]
        t_means[ip, imu] = np.mean(vals)
        t_stds[ip, imu] = np.std(vals)
        t_sems[ip, imu] = t_stds[ip, imu] / np.sqrt(n_realizations)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel A: t* vs mu
ax = axes[0]
for ip, p in enumerate(sparsity_values):
    ax.errorbar(mu_values, t_means[ip, :], yerr=t_sems[ip, :],
                fmt=f'{markers[p]}-', capsize=4, linewidth=2, markersize=8,
                color=colors_p[p], label=f'p={p}')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'Peak time $t^*$')
ax.set_title(r'(a) Peak Time vs $\mu$')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel B: t* vs mu on log-log
ax = axes[1]
for ip, p in enumerate(sparsity_values):
    ax.errorbar(mu_values, t_means[ip, :], yerr=t_sems[ip, :],
                fmt=f'{markers[p]}-', capsize=4, linewidth=2, markersize=8,
                color=colors_p[p], label=f'p={p}')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$t^*$')
ax.set_title(r'(b) Log-log: $t^* \sim \mu^{-\delta}$?')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Fit power law t* = A * mu^(-delta) using only points with nonzero variance
def power_law(mu, A, delta):
    return A * np.power(mu, -delta)

for ip, p in enumerate(sparsity_values):
    try:
        # Only fit points where t* has nonzero variance (not grid-locked)
        valid = [i for i in range(n_mu) if t_stds[ip, i] > 1e-10]
        if len(valid) < 3:
            valid = list(range(n_mu))  # fall back to all
        mu_fit_data = [mu_values[i] for i in valid]
        t_fit_data = [t_means[ip, i] for i in valid]
        t_fit_err = [max(t_sems[ip, i], 1e-6) for i in valid]
        popt, _ = curve_fit(power_law, mu_fit_data, t_fit_data,
                            p0=[1.0, 0.5], sigma=t_fit_err,
                            absolute_sigma=True)
        mu_fit = np.linspace(0.02, 0.5, 100)
        ax.plot(mu_fit, power_law(mu_fit, *popt), '--', color=colors_p[p],
                alpha=0.5, linewidth=1.5)
        print(f'  p={p:.2f}: t* = {popt[0]:.2f} * mu^(-{popt[1]:.3f})')
    except RuntimeError:
        print(f'  p={p:.2f}: power law fit failed')

# Panel C: ANOVA for peak time
ax = axes[2]
t_anova_p = []
t_grid_locked = []
for imu, mu in enumerate(mu_values):
    groups = [peak_times[ip, imu, :] for ip in range(n_sparsity)]
    # At large mu, all peak times snap to the same grid point
    is_locked = all(np.std(g) < 1e-10 for g in groups)
    t_grid_locked.append(is_locked)
    if is_locked:
        t_anova_p.append(1.0)  # identical distributions -> no difference
    else:
        try:
            _, pv = stats.f_oneway(*groups)
            t_anova_p.append(pv if not np.isnan(pv) else 1.0)
        except Exception:
            t_anova_p.append(1.0)

bar_colors_t = ['lightgray' if locked else 'C2' for locked in t_grid_locked]
ax.bar(range(len(mu_values)), [-np.log10(max(pv, 1e-20)) for pv in t_anova_p],
       color=bar_colors_t, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.axhline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.set_xticks(range(len(mu_values)))
ax.set_xticklabels([f'{m:.2f}' for m in mu_values])
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$-\log_{10}(p_{\mathrm{ANOVA}})$')
ax.set_title(r'(c) ANOVA: Is $t^*$ sparsity-dependent?')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

plt.suptitle(r'A4: Peak Time $t^*(\mu, p)$', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, '11_A4_peak_time.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print summary
dt = t_array[1] - t_array[0]
print(f'\n=== A4: Peak Time Summary (dt={dt:.3f}) ===')
print(f'{"mu":>6}', end='')
for p in sparsity_values:
    print(f'  t*(p={p:.2f})', end='')
print('  ANOVA p')
print('-' * 70)
for imu, mu in enumerate(mu_values):
    print(f'{mu:>6.2f}', end='')
    for ip in range(n_sparsity):
        print(f'  {t_means[ip, imu]:>8.2f}+/-{t_sems[ip, imu]:.2f}', end='')
    note = ' (grid-locked)' if t_grid_locked[imu] else ''
    print(f'  {t_anova_p[imu]:.3e}{note}')

n_t_sig = sum(1 for i, pv in enumerate(t_anova_p) if pv < 0.05 and not t_grid_locked[i])
n_t_locked = sum(t_grid_locked)
print(f'\nGrid-locked points (all realizations on same t-grid): {n_t_locked}/{len(mu_values)}')
print(f'Significant ANOVA (excluding grid-locked): {n_t_sig}/{len(mu_values) - n_t_locked}')
if n_t_sig > 0:
    print('Note: Marginal significance at intermediate mu likely reflects')
    print(f'discretization (dt={dt:.3f}) rather than physical sparsity dependence.')

### A5: Peak Width (FWHM) vs $\mu$ [RETRACTED]

**RETRACTION NOTICE**: The FWHM values in this section were computed using a flawed
`extract_peak` function that measured the total time span of *all* points above half-max,
including late-time quantum recurrences. This produced spuriously large and non-monotonic
FWHM values, with bimodal distributions and artifacts from recurrence crossings.

The `extract_peak` function has been corrected to search left and right from the peak
for the first crossing below half-max, with linear interpolation for sub-grid accuracy.
See `12_fwhm_corrected.ipynb` for the reanalysis with corrected FWHM.

**The FWHM results below are invalid.** The code is preserved for transparency only.

In [ ]:
# Compute FWHM statistics
w_means = np.zeros((n_sparsity, n_mu))
w_sems = np.zeros((n_sparsity, n_mu))
w_stds = np.zeros((n_sparsity, n_mu))

for ip in range(n_sparsity):
    for imu in range(n_mu):
        vals = peak_fwhms[ip, imu, :]
        valid = vals[~np.isnan(vals)]
        if len(valid) > 0:
            w_means[ip, imu] = np.mean(valid)
            w_stds[ip, imu] = np.std(valid)
            w_sems[ip, imu] = w_stds[ip, imu] / np.sqrt(len(valid))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel A: FWHM vs mu
ax = axes[0]
for ip, p in enumerate(sparsity_values):
    ax.errorbar(mu_values, w_means[ip, :], yerr=w_sems[ip, :],
                fmt=f'{markers[p]}-', capsize=4, linewidth=2, markersize=8,
                color=colors_p[p], label=f'p={p}')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel('FWHM')
ax.set_title(r'(a) Peak Width vs $\mu$')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel B: Ratio of FWHM to dense (to see if p=0.05 is systematically wider)
ax = axes[1]
for ip, p in enumerate(sparsity_values):
    if ip == 0:
        continue
    ratios = w_means[ip, :] / w_means[0, :]
    ratio_err = ratios * np.sqrt((w_sems[ip, :] / w_means[ip, :])**2 +
                                  (w_sems[0, :] / w_means[0, :])**2)
    ax.errorbar(mu_values, ratios, yerr=ratio_err,
                fmt=f'{markers[p]}-', capsize=4, linewidth=2, markersize=8,
                color=colors_p[p], label=f'p={p} / p=1.0')
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel(r'$\mu$')
ax.set_ylabel('FWHM ratio to dense')
ax.set_title('(b) FWHM Ratio: p=0.05 Wider?')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel C: ANOVA for FWHM
ax = axes[2]
w_anova_p = []
for imu, mu in enumerate(mu_values):
    groups = [peak_fwhms[ip, imu, :] for ip in range(n_sparsity)]
    groups_valid = [g[~np.isnan(g)] for g in groups]
    if all(len(g) > 1 for g in groups_valid):
        _, pv = stats.f_oneway(*groups_valid)
        w_anova_p.append(pv if not np.isnan(pv) else 1.0)
    else:
        w_anova_p.append(1.0)

bar_colors_w = ['C3' if pv < 0.05 else 'C4' for pv in w_anova_p]
ax.bar(range(len(mu_values)), [-np.log10(max(pv, 1e-20)) for pv in w_anova_p],
       color=bar_colors_w, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.axhline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.axhline(-np.log10(0.05 / len(mu_values)), color='darkred', linestyle='--',
           alpha=0.5, label='Bonferroni')
ax.set_xticks(range(len(mu_values)))
ax.set_xticklabels([f'{m:.2f}' for m in mu_values])
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$-\log_{10}(p_{\mathrm{ANOVA}})$')
ax.set_title('(c) ANOVA: FWHM Shows Some Dependence')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')

plt.suptitle(r'A5: Peak Width (FWHM) vs $\mu$ and Sparsity', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, '11_A5_peak_width.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print summary
n_w_sig = sum(1 for pv in w_anova_p if pv < 0.05)
n_w_bonf = sum(1 for pv in w_anova_p if pv < 0.05 / len(mu_values))
print(f'\n=== A5: FWHM Summary ===')
print(f'{"mu":>6}', end='')
for p in sparsity_values:
    print(f'  FWHM(p={p:.2f})', end='')
print('  ANOVA p      sig?')
print('-' * 80)
for imu, mu in enumerate(mu_values):
    print(f'{mu:>6.2f}', end='')
    for ip in range(n_sparsity):
        print(f'  {w_means[ip, imu]:>8.2f}+/-{w_sems[ip, imu]:.2f}', end='')
    sig = ' *' if w_anova_p[imu] < 0.05 else ''
    bonf = ' (Bonf)' if w_anova_p[imu] < 0.05 / len(mu_values) else ''
    print(f'  {w_anova_p[imu]:.3e}{sig}{bonf}')

print(f'\nSignificant at p<0.05: {n_w_sig}/{len(mu_values)}')
print(f'Survive Bonferroni: {n_w_bonf}/{len(mu_values)}')
print()

# Characterize the direction of the effect
print('Direction of effect (FWHM ratio p=0.05 / p=1.0):')
for imu, mu in enumerate(mu_values):
    ratio = w_means[2, imu] / w_means[0, imu]
    print(f'  mu={mu:.2f}: ratio = {ratio:.3f}')
mean_ratio = np.mean(w_means[2, :] / w_means[0, :])
print(f'  Mean ratio: {mean_ratio:.3f}')
print()
if n_w_sig > 0:
    print('NOTE: Unlike peak height (H0 supported), the FWHM shows')
    print('some sparsity dependence. The non-chaotic system (p=0.05)')
    print('produces systematically wider peaks. However, this is a')
    print('secondary observable; the primary signal (peak height) is')
    print('still sparsity-invariant.')

## 7. Interpretation: Which Outcome?

Three pre-registered outcomes were defined:

| Outcome | Criterion | Implication |
|---------|-----------|-------------|
| 1: H0 supported | Curves overlay at all $\mu$, no ANOVA significant | $\mu$ alone controls signal |
| 2: Weak H1 | Small deviation at low $\mu$ only | Chaos has marginal effect |
| 3: Strong H1 | Systematic separation at multiple $\mu$ | Chaos meaningfully modulates signal |

In [ ]:
# Systematic assessment
print('=== Outcome Determination ===')
print()

# Test 1: Curve overlay (primary observable: peak height)
max_diffs = []
for imu, mu in enumerate(mu_values):
    max_diff = max(abs(h_means[i, imu] - h_means[j, imu])
                   for i in range(n_sparsity) for j in range(i+1, n_sparsity))
    max_diffs.append(max_diff)
overall_max_diff = max(max_diffs)
mean_signal = np.mean(h_means)
relative_max_diff = overall_max_diff / mean_signal * 100

print(f'Test 1: Peak height curve overlay')
print(f'  Max absolute difference across all mu: {overall_max_diff:.4f}')
print(f'  Relative to mean signal: {relative_max_diff:.2f}%')
overlay_pass = relative_max_diff < 5.0  # <5% difference
print(f'  Overlay (<5%): {"PASS" if overlay_pass else "FAIL"}')
print()

# Test 2: ANOVA significance (peak height)
n_sig_nominal = sum(1 for r in anova_results if r['p'] < 0.05)
n_sig_bonf = sum(1 for r in anova_results if r['p'] < 0.05 / len(mu_values))
print(f'Test 2: Peak height ANOVA')
print(f'  Significant at p<0.05: {n_sig_nominal}/{len(mu_values)}')
print(f'  Significant after Bonferroni: {n_sig_bonf}/{len(mu_values)}')
anova_pass = n_sig_bonf == 0
print(f'  No Bonferroni-significant results: {"PASS" if anova_pass else "FAIL"}')
print()

# Determine outcome (based on primary observable only)
print('=' * 60)
if overlay_pass and anova_pass:
    outcome = 1
    print('OUTCOME 1: H0 SUPPORTED (for primary observable)')
    print()
    print('The coupling strength mu alone determines the transmission')
    print('peak height. Internal chaos (as measured by level spacing')
    print('ratio) has no statistically detectable effect on peak height')
    print('at any coupling strength from mu=0.02 to mu=0.50.')
elif n_sig_nominal > 0 and n_sig_bonf == 0:
    outcome = 2
    print('OUTCOME 2: WEAK H1')
    print()
    print('Minor sparsity dependence detected at nominal significance')
    print('but does not survive multiple-comparison correction.')
else:
    outcome = 3
    print('OUTCOME 3: STRONG H1')
    print()
    print('Systematic sparsity dependence detected at multiple mu values.')

print()
print('Note: A3 (tanh fit) and A5 (FWHM) have been retracted.')
print('  A3: chi2/dof >> 1, model rejected.')
print('  A5: FWHM extraction was flawed (captured recurrences).')
print('  See 12_fwhm_corrected.ipynb for corrected FWHM analysis.')
print()
print('Implication for the Google experiment:')
if outcome == 1:
    print('  The transmission peak does not probe quantum chaos.')
    print('  It probes the L-R coupling, which is a property of the')
    print('  experimental protocol, not the underlying physics.')
    print('  A non-chaotic system with the same mu produces the same')
    print('  peak height.')

## 8. Honest Assessment

### What This Establishes

1. **At N=10, beta=8**: The transmission peak height $H(\mu)$ is a function of $\mu$ alone,
   with no detectable dependence on whether the internal SYK Hamiltonian is chaotic
   ($\langle r \rangle \approx 0.59$) or non-chaotic ($\langle r \rangle \approx 0.39$).

2. **Statistical confidence**: 50 disorder realizations per point, 1200 total instances,
   zero failures. ANOVA p-values for peak height are well above significance thresholds
   at all 8 $\mu$ values.

3. **Peak height and peak time** both show the decoupling pattern: $\mu$-dependence
   with no sparsity modulation.

### Retracted Analyses

4. **A3 (Functional fit) RETRACTED**: The tanh fit $H(\mu) = \tanh(\alpha \mu^\gamma)$
   has chi2/dof ranging from 38 to 1785, meaning the model is statistically rejected.
   The fit parameters should not be interpreted as meaningful.

5. **A5 (FWHM) RETRACTED**: The original FWHM extraction was flawed — it measured the
   total time span of all points above half-max rather than the true peak width. This
   captured late-time quantum recurrences, producing spuriously large and non-monotonic
   values. The `extract_peak` function has been corrected and the reanalysis is in
   `12_fwhm_corrected.ipynb`.

### What This Does Not Establish

1. **Not a proof**: This is numerical evidence at finite $N$. The conclusion could
   change at larger $N$ where the SYK model approaches its solvable limit.

2. **Does not address ML-optimized sparsification**: The Google experiment used a
   learned sparsification that may have different properties than random.

3. **Temperature dependence unexplored**: We used $\beta = 8$ only. At very high
   temperature ($\beta \to 0$), the TFD approaches a maximally mixed state and the
   signal vanishes for all $\mu$. The beta-dependence could reveal chaos effects.

## 9. Limitations and Future Directions

### Limitations

1. **System size**: $N = 10$ Majoranas per side gives Hilbert space dimension 1024.
   The SYK model is exactly solvable only at $N \to \infty$, and finite-$N$ effects
   could mask chaos-dependent corrections to the signal.

2. **Sparsity range**: Only three sparsity values. Finer sampling around the
   chaos transition ($p \approx 0.07-0.10$) could reveal a narrow window where
   chaos effects emerge.

3. **Single observable**: The transmission peak is one of many possible probes.
   OTOC in the doubled system, or mutual information between L and R, may show
   different behavior.

4. **Fixed beta**: $\beta = 8$ puts the TFD deep in the entangled regime. The
   mechanism may change at weaker entanglement ($\beta \lesssim 1$).

5. **No noise**: This study is noiseless. Gap 2 showed that noise degrades the
   signal uniformly, but the interaction between noise and mu-dependence is
   unexplored.

### What Would Strengthen the Finding

1. **N=14 or N=18 mu sweep**: Repeat the full mu sweep at larger $N$ using
   Krylov time evolution. If the overlay persists, finite-size effects are ruled out.

2. **Analytical argument**: Derive the signal independence from the structure of
   the coupled Hamiltonian $H_0 + V$ and the TFD state, showing that the
   transmission amplitude depends on the eigenvalue structure of $V$ (which
   is $\mu$-dependent) rather than $H_0$ (which is sparsity-dependent).

3. **Alternative observables**: Compute mutual information $I(L:R; t)$ or
   doubled-system OTOC to test whether chaos effects appear in other probes.

4. **Beta scan**: Sweep $\beta$ at fixed $\mu$ across sparsities to identify
   any temperature regime where chaos effects emerge.

## Summary Figure

In [ ]:
# Combined summary figure (updated: FWHM and tanh fit retracted)
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# (0,0): Primary H(mu) plot
ax = fig.add_subplot(gs[0, 0])
for ip, p in enumerate(sparsity_values):
    ax.errorbar(mu_values, h_means[ip, :], yerr=h_sems[ip, :],
                fmt=f'{markers[p]}-', capsize=3, linewidth=2, markersize=7,
                color=colors_p[p], label=f'p={p}')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$H(\mu)$')
ax.set_title(r'(a) $H(\mu)$: Curves Overlay')
ax.legend(fontsize=8)
ax.set_ylim(0.3, 1.05)
ax.grid(True, alpha=0.3)

# (0,1): ANOVA p-values
ax = fig.add_subplot(gs[0, 1])
ax.bar(range(len(mu_values)), [-np.log10(r['p']) for r in anova_results],
       color='C1', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.axhline(-np.log10(0.05), color='red', linestyle='--', alpha=0.5, label='p=0.05')
ax.axhline(-np.log10(0.05 / len(mu_values)), color='darkred', linestyle='--',
           alpha=0.5, label='Bonferroni')
ax.set_xticks(range(len(mu_values)))
ax.set_xticklabels([f'{m:.2f}' for m in mu_values], fontsize=8)
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$-\log_{10}(p)$')
ax.set_title('(b) ANOVA: No Significant Differences')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')

# (0,2): Chaos verification
ax = fig.add_subplot(gs[0, 2])
r_means_plot = [np.mean(r_values[ip, :]) for ip in range(n_sparsity)]
r_sems_plot = [np.std(r_values[ip, :]) / np.sqrt(n_realizations) for ip in range(n_sparsity)]
bar_colors_plot = [colors_p[p] for p in sparsity_values]
ax.bar(range(n_sparsity), r_means_plot, yerr=r_sems_plot, capsize=6,
       color=bar_colors_plot, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.axhline(0.6027, color='red', linestyle='--', alpha=0.4, label='GUE')
ax.axhline(0.3863, color='gray', linestyle='--', alpha=0.4, label='Poisson')
ax.set_xticks(range(n_sparsity))
ax.set_xticklabels([f'p={p}' for p in sparsity_values])
ax.set_ylabel(r'$\langle r \rangle$')
ax.set_title('(c) Chaos is Destroyed (A6)')
ax.set_ylim(0, 0.75)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis='y')

# (1,0): Peak time
ax = fig.add_subplot(gs[1, 0])
for ip, p in enumerate(sparsity_values):
    ax.errorbar(mu_values, t_means[ip, :], yerr=t_sems[ip, :],
                fmt=f'{markers[p]}-', capsize=3, linewidth=2, markersize=7,
                color=colors_p[p], label=f'p={p}')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$t^*$')
ax.set_title(r'(d) Peak Time: Also Overlays')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (1,1): Zoomed low-mu region
ax = fig.add_subplot(gs[1, 1])
low_mu_idx = [i for i, m in enumerate(mu_values) if m <= 0.10]
for ip, p in enumerate(sparsity_values):
    ax.errorbar([mu_values[i] for i in low_mu_idx],
                [h_means[ip, i] for i in low_mu_idx],
                yerr=[h_sems[ip, i] for i in low_mu_idx],
                fmt=f'{markers[p]}-', capsize=5, linewidth=2.5, markersize=9,
                color=colors_p[p], label=f'p={p}')
    low_means = np.array([h_means[ip, i] for i in low_mu_idx])
    low_stds = np.array([h_stds[ip, i] for i in low_mu_idx])
    ax.fill_between([mu_values[i] for i in low_mu_idx],
                    low_means - low_stds, low_means + low_stds,
                    alpha=0.1, color=colors_p[p])
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$H(\mu)$')
ax.set_title(r'(e) Zoomed $\mu \leq 0.10$ (1$\sigma$ bands)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (1,2): Conclusion text
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('(f) Conclusion', fontsize=11)

conclusion_text = [
    (9.2, r'$\mathbf{Outcome\ 1:\ H_0\ Supported}$', 'black', 12),
    (8.0, 'Peak height: mu alone controls signal', 'green', 10),
    (7.0, 'ANOVA: 0/8 significant at any mu', 'green', 10),
    (6.0, 'Peak time: also decouples', 'green', 10),
    (4.5, 'RETRACTED:', 'red', 11),
    (3.5, '  A3: tanh fit (chi2/dof >> 1)', 'red', 9),
    (2.5, '  A5: FWHM (extraction bug)', 'red', 9),
    (1.5, '  See 12_fwhm_corrected.ipynb', 'gray', 9),
    (0.0, f'1200 instances, 0 failures, {total_time/3600:.0f}h compute', 'gray', 9),
]
for y, text, color, size in conclusion_text:
    ax.text(0.5, y, text, fontsize=size, color=color, va='center')

plt.suptitle('Mechanism of Signal-Chaos Decoupling: Summary',
             fontsize=15, y=1.02)
plt.savefig(os.path.join(results_dir, '11_mu_mechanism_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('All figures saved to results/ directory.')

In [ ]:
# Final quantitative summary (updated with retractions)
print('=' * 70)
print('QUANTITATIVE SUMMARY: MU MECHANISM INVESTIGATION')
print('=' * 70)
print()
print(f'Parameters: N={N_per_side}, beta={beta}, {n_realizations} realizations')
print(f'Compute: {total_time/3600:.1f} hours, {n_sparsity * n_mu * n_realizations} instances, {failures} failures')
print()
print('A6 (Chaos verification):')
for ip, p in enumerate(sparsity_values):
    rv = r_values[ip, :]
    print(f'  p={p:.2f}: <r> = {np.mean(rv):.4f} +/- {np.std(rv)/np.sqrt(len(rv)):.4f} ({labels_p[p]})')
print()
print('A1 (H(mu) overlay):')
print(f'  Max deviation between curves: {overall_max_diff:.4f} ({relative_max_diff:.2f}%)')
print(f'  At weakest coupling (mu={mu_values[0]}): ', end='')
for ip in range(n_sparsity):
    print(f'  H={h_means[ip, 0]:.4f}', end='')
print()
print(f'  At strongest coupling (mu={mu_values[-1]}): ', end='')
for ip in range(n_sparsity):
    print(f'  H={h_means[ip, -1]:.4f}', end='')
print()
print()
print('A2 (ANOVA on peak height):')
print(f'  Significant at p<0.05: {n_sig_nominal}/{len(mu_values)}')
print(f'  Survive Bonferroni: {n_sig_bonf}/{len(mu_values)}')
print()
print('A3 (Functional fit): *** RETRACTED ***')
print('  Reason: chi2/dof >> 1 at all sparsities (38 to 1785)')
print('  The tanh model is statistically rejected.')
print()
print('A4 (Peak time):')
print(f'  Curves overlay; discretization-dominated at large mu')
print()
print('A5 (FWHM): *** RETRACTED ***')
print('  Reason: extract_peak measured total span above half-max,')
print('  not true peak width. Captured late-time recurrences.')
print('  Corrected analysis in 12_fwhm_corrected.ipynb.')
print()
print('=' * 70)
print('VERDICT: Outcome 1 (H0 supported for peak height)')
print()
print('  The L-R coupling mu alone controls the transmission peak height.')
print('  Internal quantum chaos has no detectable effect on peak amplitude.')
print()
print('  Remaining valid analyses: A1, A2, A4, A6.')
print('  Retracted analyses: A3 (tanh fit), A5 (FWHM).')
print('  See 12_fwhm_corrected.ipynb for corrected FWHM results.')
print('=' * 70)